In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# ─────────────────────────────────────
# STEP 1 — Load enriched data
# ─────────────────────────────────────
# Using enriched file that has churn_prob, clv, strategy etc.
df = pd.read_csv('../data/bank_churn_with_predictions.csv')
print(f'Total customers: {len(df)}')
print(f'Columns available: {list(df.columns)}')

# ─────────────────────────────────────
# STEP 2 — Select at-risk customers
# ─────────────────────────────────────
at_risk = df[df['churn_prob'] > 0.50].copy()
print(f'\nAt-risk customers (>50% churn prob): {len(at_risk)}')
print(f'Safe customers: {len(df) - len(at_risk)}')

# ─────────────────────────────────────
# STEP 3 — Randomly assign groups
# ─────────────────────────────────────
np.random.seed(42)
at_risk['group'] = np.random.choice(
    ['control', 'treatment'],
    size=len(at_risk),
    p=[0.5, 0.5]
)
print(f'\nGroup split:')
print(at_risk['group'].value_counts())

# ─────────────────────────────────────
# STEP 4 — Simulate churn outcome
# ─────────────────────────────────────
def simulate_churn(row):
    base_rate = row['churn_prob']
    if row['group'] == 'treatment':
        # Strategy reduces churn probability
        effective_rate = base_rate * (1 - row['retention_lift'])
    else:
        # Control group gets no intervention
        effective_rate = base_rate
    return np.random.binomial(1, effective_rate)

at_risk['churned_simulated'] = at_risk.apply(simulate_churn, axis=1)

# ─────────────────────────────────────
# STEP 5 — Summary
# ─────────────────────────────────────
summary = at_risk.groupby('group')['churned_simulated'].agg(['mean', 'sum', 'count'])
summary.columns = ['Churn Rate', 'Total Churned', 'Total Customers']
print('\n--- A/B Test Summary ---')
print(summary)

# ─────────────────────────────────────
# STEP 6 — Statistical significance
# ─────────────────────────────────────
contingency = pd.crosstab(
    at_risk['group'],
    at_risk['churned_simulated']
)
print('\nContingency Table:')
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)

ctrl_rate  = at_risk[at_risk['group'] == 'control']['churned_simulated'].mean()
treat_rate = at_risk[at_risk['group'] == 'treatment']['churned_simulated'].mean()
reduction  = (ctrl_rate - treat_rate) / ctrl_rate * 100

print(f'\n--- Statistical Test Results ---')
print(f'Control churn rate:   {ctrl_rate:.1%}')
print(f'Treatment churn rate: {treat_rate:.1%}')
print(f'Churn reduction:      {reduction:.1f}%')
print(f'Chi2:                 {chi2:.2f}')
print(f'P-value:              {p_value:.4f}')
print(f'Result: {"✅ SIGNIFICANT" if p_value < 0.05 else "❌ NOT Significant"}')

# ─────────────────────────────────────
# STEP 7 — ROI Calculation
# ─────────────────────────────────────
clv_saved_treatment = at_risk[
    (at_risk['group'] == 'treatment') &
    (at_risk['churned_simulated'] == 0)
]['clv'].sum()

clv_saved_control = at_risk[
    (at_risk['group'] == 'control') &
    (at_risk['churned_simulated'] == 0)
]['clv'].sum()

total_intervention_cost = at_risk[
    at_risk['group'] == 'treatment'
]['intervention_cost'].sum()

net_benefit = (clv_saved_treatment - clv_saved_control) - total_intervention_cost
roi_pct     = (net_benefit / total_intervention_cost * 100 
               if total_intervention_cost > 0 else 0)

print(f'\n--- ROI Calculation ---')
print(f'CLV saved (treatment): ${clv_saved_treatment:,.0f}')
print(f'CLV saved (control):   ${clv_saved_control:,.0f}')
print(f'Intervention cost:     ${total_intervention_cost:,.0f}')
print(f'Net benefit:           ${net_benefit:,.0f}')
print(f'ROI:                   {roi_pct:.1f}%')
print(f'Annualized (x12):      ${net_benefit * 12:,.0f}/year')

# ─────────────────────────────────────
# STEP 8 — Strategy breakdown
# ─────────────────────────────────────
print(f'\n--- Strategy Distribution ---')
print(at_risk.groupby('strategy')['churned_simulated'].agg(['mean', 'count'])
      .rename(columns={'mean': 'Churn Rate', 'count': 'Customers'})
      .round(3))

Total customers: 10000
Columns available: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'balance_salary_ratio', 'products_per_tenure', 'is_zero_balance', 'age_x_inactive', 'credit_per_age', 'high_bal_inactive', 'segment', 'clv', 'churn_prob', 'strategy', 'intervention_cost', 'retention_lift', 'expected_value', 'priority']

At-risk customers (>50% churn prob): 2491
Safe customers: 7509

Group split:
group
treatment    1257
control      1234
Name: count, dtype: int64

--- A/B Test Summary ---
           Churn Rate  Total Churned  Total Customers
group                                                
control      0.754457            931             1234
treatment    0.498807            627             1257

Contingency Table:
churned_simulated    0    1
group                      
control            303  931
treatment          630  627

--- Statistical Test Results ---
Control churn rate:   75

In [2]:
# ─────────────────────────────────────
# FULL DOLLAR ROI CALCULATION
# ─────────────────────────────────────

# STEP 1 — CLV saved in each group
clv_saved_treatment = at_risk[
    (at_risk['group'] == 'treatment') &
    (at_risk['churned_simulated'] == 0)
]['clv'].sum()

clv_saved_control = at_risk[
    (at_risk['group'] == 'control') &
    (at_risk['churned_simulated'] == 0)
]['clv'].sum()

# STEP 2 — Total intervention cost
total_intervention_cost = at_risk[
    at_risk['group'] == 'treatment'
]['intervention_cost'].sum()

# STEP 3 — Net benefit and ROI
net_benefit = (clv_saved_treatment - clv_saved_control) - total_intervention_cost
roi_pct     = net_benefit / total_intervention_cost * 100

# STEP 4 — Churners saved count
churners_saved = at_risk[
    (at_risk['group'] == 'treatment') &
    (at_risk['churned_simulated'] == 0)
].shape[0] - at_risk[
    (at_risk['group'] == 'control') &
    (at_risk['churned_simulated'] == 0)
].shape[0]

# STEP 5 — Cost per churner saved
cost_per_churner = (total_intervention_cost / churners_saved 
                    if churners_saved > 0 else 0)

# STEP 6 — Strategy level ROI breakdown
strategy_roi = at_risk[at_risk['group'] == 'treatment'].groupby('strategy').apply(
    lambda x: pd.Series({
        'Customers Targeted':    len(x),
        'Churners Saved':        (x['churned_simulated'] == 0).sum(),
        'Total Cost':            x['intervention_cost'].sum(),
        'CLV Saved':             x[x['churned_simulated'] == 0]['clv'].sum(),
        'Net Benefit':           x[x['churned_simulated'] == 0]['clv'].sum() - x['intervention_cost'].sum(),
        'ROI %':                 ((x[x['churned_simulated'] == 0]['clv'].sum() - 
                                   x['intervention_cost'].sum()) / 
                                   x['intervention_cost'].sum() * 100
                                   if x['intervention_cost'].sum() > 0 else 0)
    })
).round(1)

# ─────────────────────────────────────
# PRINT FULL REPORT
# ─────────────────────────────────────
print('=' * 55)
print('         FULL DOLLAR ROI REPORT')
print('=' * 55)

print(f'\n📊 GROUP PERFORMANCE:')
print(f'   CLV saved (treatment):    ${clv_saved_treatment:>12,.0f}')
print(f'   CLV saved (control):      ${clv_saved_control:>12,.0f}')
print(f'   Extra CLV from treatment: ${clv_saved_treatment - clv_saved_control:>12,.0f}')

print(f'\n💰 COST ANALYSIS:')
print(f'   Total intervention cost:  ${total_intervention_cost:>12,.0f}')
print(f'   Churners saved:           {churners_saved:>12}')
print(f'   Cost per churner saved:   ${cost_per_churner:>12,.0f}')

print(f'\n🏆 BOTTOM LINE:')
print(f'   Net benefit:              ${net_benefit:>12,.0f}')
print(f'   ROI:                      {roi_pct:>11.1f}%')
print(f'   Monthly benefit:          ${net_benefit:>12,.0f}')
print(f'   Annualized (x12):         ${net_benefit * 12:>12,.0f}/year')

print(f'\n📈 FOR EVERY $1 SPENT:')
print(f'   Bank gets back:           ${roi_pct/100 + 1:>12.1f}')

print('\n' + '=' * 55)
print('   STRATEGY LEVEL ROI BREAKDOWN')
print('=' * 55)
print(strategy_roi[['Customers Targeted', 'Churners Saved', 
                     'Total Cost', 'CLV Saved', 
                     'Net Benefit', 'ROI %']].to_string())

print('\n' + '=' * 55)
print('   EXECUTIVE SUMMARY')
print('=' * 55)
print(f'''
   Problem:   Bank losing customers to churn
   Solution:  ML model + targeted interventions
   Test:      A/B test on {len(at_risk):,} at-risk customers
   
   Results:
   → Churn reduced from 75.4% to 49.9% (-33.9%)
   → {churners_saved} additional customers retained
   → ${net_benefit:,.0f} net benefit this period
   → ${net_benefit * 12:,.0f} projected annual saving
   → {roi_pct:.0f}% ROI on intervention budget
   → P-value: 0.0000 (statistically proven ✅)
   
   Recommendation: 
   → Roll out to ALL at-risk customers immediately!
''')

         FULL DOLLAR ROI REPORT

📊 GROUP PERFORMANCE:
   CLV saved (treatment):    $   6,481,026
   CLV saved (control):      $   2,741,075
   Extra CLV from treatment: $   3,739,951

💰 COST ANALYSIS:
   Total intervention cost:  $      75,602
   Churners saved:                    327
   Cost per churner saved:   $         231

🏆 BOTTOM LINE:
   Net benefit:              $   3,664,349
   ROI:                           4846.9%
   Monthly benefit:          $   3,664,349
   Annualized (x12):         $  43,972,183/year

📈 FOR EVERY $1 SPENT:
   Bank gets back:           $        49.5

   STRATEGY LEVEL ROI BREAKDOWN
                                 Customers Targeted  Churners Saved  Total Cost  CLV Saved  Net Benefit     ROI %
strategy                                                                                                         
Dedicated Relationship Manager                119.0            65.0     14280.0  1145048.2    1130768.2    7918.5
Email + Fee Waiver                    

C:\Users\aishw\AppData\Local\Temp\ipykernel_6880\2686408357.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  strategy_roi = at_risk[at_risk['group'] == 'treatment'].groupby('strategy').apply(
